## ENTREGABLE 1 - HERRERA TORRES, DANIEL

El código de funcionamiento erróneo, al definir la función genérica get_SOS_help, obtiene ayuda de Stack Overflow usando web scrapping, proporcionando la respuesta con más votos de los usuarios para el error específico que encuentra en el código. 

El output del código muestra como se comporta el código en caso de estar correcto y en caso de tener algún error.

In [ ]:
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

def get_SOS_help(error_message):
    """
    Busca ayuda en Stack Overflow para el error proporcionado.
    Permite que el usuario resuelva captchas manualmente.
    Obtiene únicamente el texto principal de la respuesta más votada.
    """
    # -----------------------
    # CONFIGURACIÓN SELENIUM
    # -----------------------
    # Creamos un objeto de configuración para Chrome. Añadimos argumentos
    # para evitar problemas de sandboxing y para iniciar la ventana maximizada.
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")            # Evita problemas de sandbox en entornos como Docker o Colab.
    chrome_options.add_argument("--disable-dev-shm-usage") # Reduce problemas de memoria compartida en algunos entornos.
    chrome_options.add_argument("--start-maximized")       # Inicia el navegador en pantalla maximizada.
    chrome_options.add_argument("--disable-extensions")    # Desactiva extensiones de Chrome.

    # Iniciamos el navegador Chrome con las opciones anteriores.
    driver = webdriver.Chrome(options=chrome_options)

    try:
        # -----------------------
        # PASO 1: ABRIR GOOGLE
        # -----------------------
        # Navega a Google, donde haremos la búsqueda del error.
        driver.get("https://www.google.com")

        # -----------------------
        # PASO 2: ACEPTAR COOKIES
        # -----------------------
        # Intentamos encontrar y hacer clic en el botón de "Aceptar todo" de cookies, si aparece.
        try:
            accept_cookies = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'Aceptar todo')]"))
            )
            accept_cookies.click()
            print("Cookies aceptadas.")
        except Exception:
            # Si no aparece el cuadro de cookies, no detenemos el flujo; solo reportamos que no se vio.
            print("No apareció el cuadro de cookies.")

        # -----------------------
        # PASO 3: BUSCAR EL ERROR
        # -----------------------
        # Buscamos la caja de texto de Google para introducir la consulta:
        # "error_message site:stackoverflow.com"
        search_box = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.NAME, "q"))
        )
        # Hacemos scroll hasta el elemento, por si no está en pantalla.
        driver.execute_script("arguments[0].scrollIntoView();", search_box)
        
        # Armamos la consulta: incluimos el mensaje de error + "site:stackoverflow.com"
        # para que Google busque exclusivamente en Stack Overflow.
        search_query = f"{error_message} site:stackoverflow.com"

        # Escribimos la consulta en el cuadro de búsqueda y presionamos Enter.
        search_box.send_keys(search_query)
        search_box.send_keys(Keys.RETURN)

        # -----------------------
        # PASO 4: MANEJO DE CAPTCHA
        # -----------------------
        # En ocasiones, Google puede mostrar un CAPTCHA.
        # Le damos al usuario 300 segundos para resolverlo manualmente si aparece.
        print("Si aparece un CAPTCHA, resuélvelo manualmente para continuar.")
        WebDriverWait(driver, 300).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a h3"))
        )

        # -----------------------
        # PASO 5: ABRIR PRIMER RESULTADO
        # -----------------------
        # Esperamos a que el primer resultado de búsqueda sea clicable, luego hacemos clic en él.
        first_result = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "a h3"))
        )
        first_result.click()

        # -----------------------
        # PASO 6: EXTRAER RESPUESTA
        # -----------------------
        # En la página de Stack Overflow, encontramos la respuesta más votada.
        most_voted_answer = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.answercell div.js-post-body"))
        )
        
        # El texto de la respuesta puede incluir cosas como "Share", "Improve this answer", etc.
        # Dividimos la cadena por la palabra "Share" y tomamos la primera parte para obtener solo el texto principal.
        answer_text = most_voted_answer.text.split("Share")[0].strip()
        
        print("Respuesta más votada:")
        print(answer_text)

    except Exception as e:
        # Si algo falla en el proceso, informamos el error.
        print(f"Error al buscar en Google o en Stack Overflow: {str(e)}")

    finally:
        # -----------------------
        # PASO 7: MANTENER VENTANA ABIERTA
        # -----------------------
        # Dejamos el navegador abierto de manera indefinida para que el usuario
        # pueda revisar la página. Si el usuario cierra la ventana, el script eventualmente
        # dejará de ejecutarse. 
        print("Mantén la ventana abierta hasta que decidas cerrarla.")
        while True:
            time.sleep(1)

def execute_command(command):
    """
    Intenta ejecutar el comando proporcionado.
    Si ocurre un error, busca ayuda en Stack Overflow.
    """
    try:
        # Ejecuta el comando usando 'eval'. Esto convierte la cadena en código Python.
        # Aquí, se asume que 'np' (numpy) ya está importado.
        result = eval(command)
        
        # Imprimimos el resultado que generó el comando, por ejemplo,
        # un arreglo de números aleatorios.
        print(f"Resultado: {result}")
        # Si todo sale bien, mostramos "Works as intended".
        print("Works as intended")
    except Exception as e:
        # Si ocurre un error en la ejecución del comando, se captura aquí.
        print(f"Ocurrió un error: {str(e)}")
        # Se llama a la función get_SOS_help para buscar la respuesta más votada en Stack Overflow.
        get_SOS_help(str(e))

# EJEMPLOS DE USO

# Comando con error intencional (se usa 'pp' en lugar de 'np', lo que causará un NameError).
command_with_error = 'pp.random.uniform(-1, 1, size=100)'  # Error intencional

# Comando correcto, genera 100 números aleatorios entre -1 y 1.
command_correct = 'np.random.uniform(-1, 1, size=100)'

# Llamamos a la función que ejecuta ambos comandos para demostrar el flujo.
execute_command(command_correct)      # Ejecutará correctamente, imprime resultado y "Works as intended".
execute_command(command_with_error)   # Generará un error y buscará la respuesta más votada en Stack Overflow.

Resultado: [ 0.83329801  0.95980063  0.91511552  0.21088278 -0.84969224  0.22809227
  0.08835233 -0.41671606  0.09345191 -0.66848395 -0.79783686  0.68764827
 -0.36799544  0.81387932 -0.72435871  0.16171442  0.95196341  0.43624357
 -0.39941727  0.51631759  0.90983313 -0.5665101   0.07956123  0.89332855
  0.9560239   0.90411042 -0.52202774  0.02024863  0.41306692 -0.40738375
 -0.79493751 -0.59919693 -0.23048306 -0.34103323 -0.87732322 -0.60164872
 -0.5626088   0.31467363 -0.882533    0.085368   -0.14665952 -0.19509089
  0.02974264  0.89249304 -0.05361159  0.03214459 -0.18295907  0.80390188
 -0.46496212  0.46331543 -0.33434595  0.17293826 -0.05979964 -0.89998213
  0.53477359 -0.6197755   0.72868622 -0.06723062  0.33095333  0.72630476
 -0.81371223  0.80805388 -0.69941665  0.22330411 -0.83140004 -0.93416526
  0.58230829  0.88480757  0.49049469  0.34101177 -0.14324934  0.92016944
  0.64626436  0.88452607 -0.13887954  0.35500771 -0.96249946  0.21758389
 -0.38527782  0.3103869  -0.74017036  0.

#### Problemas encontrados desarrollando el trabajo:

1- ElementNotInteractableException (elemento no interactuable): Ocurrió cuando Selenium intentaba enviar teclas o hacer clic en un elemento (como el buscador de Google) antes de que estuviera completamente listo o visible en la página.

2- Error “name 'np' is not defined”: Surgió al usar np.random.uniform sin importar NumPy como np o porque el entorno donde se ejecutaba el comando no conocía el alias np. Error básico pero fácil de corregir importando NumPy como np.

3- Manejo manual del CAPTCHA y cierre del navegador: Se pretende resolver el CAPTCHA manualmente en lugar de saltárselo automáticamente, y además se pretende que el navegador permanezca abierto en Stack Overflow. Ajustar el código para que no cerrara el navegador al final fue otro de los requerimientos (en vez de driver.quit(), se mantuvo un bucle infinito hasta el cierre manual).

#### Conclusiones

- La importancia de manejar correctamente los eventos e interacciones en Selenium. Se concluyo que no basta con ubicar un elemento en el DOM; debemos asegurarnos de que sea visible e interactuable, usando esperas explícitas (como WebDriverWait) y condiciones de visibilidad o clicabilidad. Así evitamos errores como ElementNotInteractableException.

- La fragilidad de los selectores y la dependencia de la estructura de la página. Al usar web scraping, dependemos de la estructura HTML y los selectores. Si la página cambia (por ejemplo, el botón de cookies o la clase de la respuesta de Stack Overflow), nuestro código deja de funcionar. Este proyecto muestra lo crítico que es tener un mantenimiento continuo y usar estrategias de scraping flexibles.